In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Environment Setup and Global Configuration

In [3]:

!pip install --quiet lightgbm

import os, random
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score
import lightgbm as lgb

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

# REPRODUCIBILITY
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# DATA & MODEL CONFIG
SEQ_LEN = 100   # window length in time steps
STEP    = 50    # stride between consecutive windows

DATA_DIR   = "/content/drive/MyDrive/[2025-2026] AN2DL/Challenge 1/Dataset"
train_path  = f"{DATA_DIR}/pirate_pain_train.csv"
labels_path = f"{DATA_DIR}/pirate_pain_train_labels.csv"
test_path   = f"{DATA_DIR}/pirate_pain_test.csv"


TensorFlow version: 2.19.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# Data Loading and Preprocessing

In [4]:

# Load data + preprocessing
#   - manual category mapping
#   - label encoding
#   - outlier clipping
#   - global standardization

# LOAD CSVs
train  = pd.read_csv(train_path)
labels = pd.read_csv(labels_path)
test   = pd.read_csv(test_path)

# Merge labels into the train time series
data = train.merge(labels, on="sample_index")

# MANUAL CATEGORY MAPPING
# Map 'one'/'two'/'three' → 1/2/3. Unknowns → 0.
cat_map = {"one": 1, "two": 2, "three": 3}
for col in ["n_legs", "n_hands", "n_eyes"]:
    data[col] = data[col].map(cat_map).fillna(0)
    test[col] = test[col].map(cat_map).fillna(0)

# LABEL ENCODING
# Encode labels to integers {0,1,2} for Keras
le = LabelEncoder()
data["label_idx"] = le.fit_transform(data["label"])
n_classes = len(le.classes_)
print("Classes:", le.classes_)

# OUTLIER CLIPPING ON JOINT FEATURES
# Clip extreme sensor values (1st–99th percentile) to stabilize training.
joint_cols = [c for c in train.columns if c.startswith("joint_")]
for col in joint_cols:
    lo = data[col].quantile(0.01)
    hi = data[col].quantile(0.99)
    data[col] = data[col].clip(lo, hi)
    test[col] = test[col].clip(lo, hi)

# GLOBAL SCALING
# Standardize all non-index / non-label features with a single scaler.
feature_cols = [c for c in data.columns if c not in ["sample_index", "label", "label_idx"]]

scaler = StandardScaler()
data[feature_cols] = scaler.fit_transform(data[feature_cols].astype(np.float32))
test[feature_cols] = scaler.transform(test[feature_cols].astype(np.float32))

print("Num features:", len(feature_cols))


Classes: ['high_pain' 'low_pain' 'no_pain']
Num features: 39


# Windowing + class weights + model definition

In [5]:

# Window generation + class weights + Keras model

# WINDOWING FUNCTION
def make_windows(df: pd.DataFrame, labeled: bool = True):
    """
    Convert variable-length sequences into fixed-length windows.
    - SEQ_LEN: length of each window (in time steps)
    - STEP: stride for the starting index
    Returns:
      X: (num_windows, SEQ_LEN, num_features)
      y: (num_windows,) window labels (if labeled=True)
      G: (num_windows,) sample_index for each window (used for GroupKFold)
    """
    X, Y, G = [], [], []
    for sid, chunk in df.groupby("sample_index"):
        arr = chunk[feature_cols].values.astype(np.float32)
        label = chunk["label_idx"].iloc[0] if labeled else None
        L = len(arr)

        # Starting indices for windows; at least one window per sample
        starts = list(range(0, max(1, L - SEQ_LEN + 1), STEP))
        if not starts:
            starts = [0]

        for s in starts:
            seg = arr[s:s+SEQ_LEN]

            # Zero-pad last window if shorter than SEQ_LEN
            if len(seg) < SEQ_LEN:
                pad = np.zeros((SEQ_LEN - len(seg), arr.shape[1]), np.float32)
                seg = np.vstack([seg, pad])

            X.append(seg)
            G.append(sid)
            if labeled:
                Y.append(label)

    if labeled:
        return np.array(X), np.array(Y), np.array(G)
    else:
        return np.array(X), None, np.array(G)

# Build windowed train & test sets
X, y, groups = make_windows(data, labeled=True)
Xt, _, gt    = make_windows(test,  labeled=False)

n_features = X.shape[2]
print("Train windows:", X.shape)
print("Test windows:", Xt.shape)

# CLASS WEIGHTS (to handle label imbalance)
win_counts = np.bincount(y, minlength=n_classes)
# Inverse-frequency style weights, normalized to mean 1
cw = win_counts.sum() / (n_classes * np.maximum(win_counts, 1))
cw /= cw.mean()
class_weights = {i: float(cw[i]) for i in range(n_classes)}
print("Class weights:", class_weights)

# CNN → CNN → BiGRU → Attention MODEL
def build_model():
    """
    CNN–BiGRU–Attention classifier.
    - Conv1D layers: local pattern extraction
    - BiGRU: temporal modeling
    - Attention: focus on most informative time steps
    """
    inp = layers.Input(shape=(SEQ_LEN, n_features))

    # Local pattern extraction
    x = layers.Conv1D(64, 5, padding="same", activation="relu")(inp)
    x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)

    # Long-term temporal modeling
    x = layers.Bidirectional(
        layers.GRU(128, return_sequences=True, dropout=0.2)
    )(x)

    # Attention: Dense → Softmax over time → weighted sum
    attn_scores  = layers.Dense(1)(x)                  # (B, T, 1)
    attn_weights = layers.Softmax(axis=1)(attn_scores) # (B, T, 1)
    weighted     = layers.Multiply()([attn_weights, x])
    context      = layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(weighted)  # (B, 2*GRU)

    # Final classifier head
    x = layers.Dense(128, activation="relu")(context)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

print("Keras model ready.")


Train windows: (1322, 100, 39)
Test windows: (2648, 100, 39)
Class weights: {0: 1.759186463852333, 1: 1.0480259784652197, 2: 0.19278755768244746}
Keras model ready.


# 5-fold GroupKFold training + deep ensemble predictions

In [6]:

# 5-fold GroupKFold training + deep ensemble inference

kf = GroupKFold(n_splits=5)
fold_models = []
f1_scores   = []

print("\nStarting 5-fold GroupKFold training...")

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y, groups)):
    print(f"\n====== FOLD {fold+1} / 5 ======")
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    model = build_model()

    cb = [
        callbacks.EarlyStopping(
            patience=7, restore_best_weights=True
        ),
        callbacks.ReduceLROnPlateau(
            patience=3, factor=0.5, verbose=1
        )
    ]

    model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=40,
        batch_size=64,
        class_weight=class_weights,
        callbacks=cb,
        verbose=1
    )

    # Window-level predictions on validation
    val_pred = model.predict(X_va, batch_size=128, verbose=0)
    fold_f1 = f1_score(y_va, val_pred.argmax(1), average="macro")
    print("Fold macro-F1 (window level):", fold_f1)

    f1_scores.append(fold_f1)
    fold_models.append(model)

print("\nF1 by fold:", f1_scores)
print("Mean F1 deep (window-level):", np.mean(f1_scores))

# DEEP ENSEMBLE ON TEST WINDOWS
# Weight folds by their validation F1 to form a weighted ensemble.
weights = np.array(f1_scores) / np.sum(f1_scores)

deep_pred = np.zeros((len(Xt), n_classes), np.float32)
for w, m in zip(weights, fold_models):
    deep_pred += w * m.predict(Xt, batch_size=128, verbose=0)

# AGGREGATE WINDOWS → SAMPLE-LEVEL PROBS
uniq = np.unique(gt)  # unique sample_index values in test
deep_sample_probs = {
    sid: deep_pred[gt == sid].mean(0) for sid in uniq
}

print("Computed deep ensemble probabilities for", len(uniq), "test samples.")



Starting 5-fold GroupKFold training...

====== FOLD 1 / 5 ======
Epoch 1/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 132ms/step - accuracy: 0.6644 - loss: 0.4355 - val_accuracy: 0.7030 - val_loss: 0.8000 - learning_rate: 0.0010
Epoch 2/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.7776 - loss: 0.3246 - val_accuracy: 0.8233 - val_loss: 0.5880 - learning_rate: 0.0010
Epoch 3/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8419 - loss: 0.2376 - val_accuracy: 0.7820 - val_loss: 0.5568 - learning_rate: 0.0010
Epoch 4/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.8834 - loss: 0.1479 - val_accuracy: 0.8083 - val_loss: 0.4702 - learning_rate: 0.0010
Epoch 5/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8966 - loss: 0.1229 - val_accuracy: 0.8271 - val_loss: 0.4706 - learning_rate: 0.0010
Epoch 6/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9297 - loss: 0.1041 - val_accuracy: 0.8571 - val_loss: 0.4853 - learning_rate: 0.0010
Epoch 7/40
17/17 ━━━━━━━━━

Fold macro-F1 (window level): 0.6294978283404847

====== FOLD 4 / 5 ======
Epoch 1/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 4s 50ms/step - accuracy: 0.5322 - loss: 0.4454 - val_accuracy: 0.6515 - val_loss: 0.8786 - learning_rate: 0.0010
Epoch 2/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7248 - loss: 0.3363 - val_accuracy: 0.7614 - val_loss: 0.7214 - learning_rate: 0.0010
Epoch 3/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8278 - loss: 0.2399 - val_accuracy: 0.7727 - val_loss: 0.6403 - learning_rate: 0.0010
Epoch 4/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8584 - loss: 0.1639 - val_accuracy: 0.8106 - val_loss: 0.5952 - learning_rate: 0.0010
Epoch 5/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8900 - loss: 0.1159 - val_accuracy: 0.8220 - val_loss: 0.5729 - learning_rate: 0.0010
Epoch 6/40
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9243 - loss: 0.0855 - val_accuracy: 0.8674 - val_loss: 0.5324 - learning_rate: 0.0010
Epoch 7/40
17/17 ━━

# Global stats features + LightGBM + blending + CSVs

In [7]:

# Global stats + LightGBM + blending + CSV export

# GLOBAL STATISTICS FEATURE ENGINEERING
def make_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-sample global statistics:
      - mean, std, min, max of each feature over time.
    Returns a DataFrame with one row per sample_index.
    """
    rows = []
    for sid, g in df.groupby("sample_index"):
        row = {"sample_index": sid}
        arr = g[feature_cols].values
        for i, c in enumerate(feature_cols):
            col = arr[:, i]
            row[f"{c}_mean"] = col.mean()
            row[f"{c}_std"]  = col.std()
            row[f"{c}_min"]  = col.min()
            row[f"{c}_max"]  = col.max()
        rows.append(row)
    return pd.DataFrame(rows)

# Compute stats for train and test sequences
train_stats = make_stats(data).merge(labels, on="sample_index")
train_stats["label_idx"] = le.transform(train_stats["label"])
test_stats  = make_stats(test)

Xg      = train_stats.drop(["sample_index", "label", "label_idx"], axis=1)
yg      = train_stats["label_idx"]
Xg_test = test_stats.drop(["sample_index"], axis=1)

print("Global stats train shape:", Xg.shape)
print("Global stats test shape :", Xg_test.shape)

# LIGHTGBM TRAINING
# Gradient-boosted trees on global statistics as a complementary model.
lgbm = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=n_classes,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8
)
lgbm.fit(Xg, yg)

lgb_probs = lgbm.predict_proba(Xg_test)  # (num_test_samples, n_classes)

# Map LightGBM outputs by sample_index
lgb_map = {
    sid: lgb_probs[i] for i, sid in enumerate(test_stats["sample_index"])
}

# BLENDING: DEEP + LGBM
# For each alpha, blend:
#   P_final = alpha * P_deep + (1 - alpha) * P_LGBM

for alpha in [0.8, 0.7, 0.6]:
    final_rows = []
    for sid in uniq:
        deep_vec = deep_sample_probs[sid]
        lgb_vec  = lgb_map[sid]
        blend    = alpha * deep_vec + (1.0 - alpha) * lgb_vec
        label    = le.inverse_transform([blend.argmax()])[0]
        final_rows.append((sid, label))

    sub = pd.DataFrame(final_rows, columns=["sample_index", "label"])
    path = f"/content/submission_a{int(alpha*100)}.csv"
    sub.to_csv(path, index=False)
    print(f"Saved blended submission (alpha={alpha}):", path)


Global stats train shape: (661, 156)
Global stats test shape : (1324, 156)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001176 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25218
[LightGBM] [Info] Number of data points in the train set: 661, number of used features: 120
[LightGBM] [Info] Start training from score -2.468402
[LightGBM] [Info] Start training from score -1.950459
[LightGBM] [Info] Start training from score -0.257384
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No 

In [8]:

# FINAL GLOBAL METRICS (using Out-Of-Fold predictions)
# Computes ONE final result: Accuracy, Precision, Recall, F1, ROC-AUC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Make OOF predictions
oof_probs = np.zeros((len(X), n_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y, groups)):
    model = fold_models[fold]
    preds = model.predict(X[va_idx], batch_size=128, verbose=0)
    oof_probs[va_idx] = preds

# Window-level → sample-level aggregation
sample_true = []
sample_pred = []
sample_prob = []

gid = groups  # window → sample_index mapping
unique_ids = np.unique(gid)

for sid in unique_ids:
    idxs = np.where(gid == sid)[0]
    true_label = y[idxs[0]]  # true label for this sample
    mean_prob  = oof_probs[idxs].mean(axis=0)

    sample_true.append(true_label)
    sample_pred.append(mean_prob.argmax())
    sample_prob.append(mean_prob)

sample_true = np.array(sample_true)
sample_pred = np.array(sample_pred)
sample_prob = np.array(sample_prob)

# Compute final global metrics
acc  = accuracy_score(sample_true, sample_pred)
prec = precision_score(sample_true, sample_pred, average="macro")
rec  = recall_score(sample_true, sample_pred, average="macro")
f1   = f1_score(sample_true, sample_pred, average="macro")

try:
    auc = roc_auc_score(
        tf.keras.utils.to_categorical(sample_true, n_classes),
        sample_prob,
        average="macro",
        multi_class="ovr"
    )
except:
    auc = float("nan")

print("\n==================== FINAL GLOBAL METRICS ====================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("==============================================================")



==================== FINAL GLOBAL METRICS ====================
Accuracy : 0.8896
Precision: 0.7575
Recall   : 0.7875
F1-score : 0.7711
ROC-AUC  : 0.9332


# OOF predictions for LightGBM (sample level)

In [9]:

# OOF predictions for LightGBM (sample level)

from sklearn.model_selection import KFold

n_samples = Xg.shape[0]
lgb_oof_probs = np.zeros((n_samples, n_classes), dtype=np.float32)

kf_lgb = KFold(n_splits=5, shuffle=True, random_state=SEED)

print("Building LightGBM OOF predictions...")

for fold, (tr_idx, va_idx) in enumerate(kf_lgb.split(Xg, yg)):
    print(f"  LGB Fold {fold+1}/5")
    X_tr, X_va = Xg.iloc[tr_idx], Xg.iloc[va_idx]
    y_tr, y_va = yg.iloc[tr_idx], yg.iloc[va_idx]

    lgbm_fold = lgb.LGBMClassifier(
        objective="multiclass",
        num_class=n_classes,
        n_estimators=500,
        learning_rate=0.05,
        max_depth=8,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED + fold,
    )
    lgbm_fold.fit(X_tr, y_tr)

    lgb_oof_probs[va_idx] = lgbm_fold.predict_proba(X_va)

print("Done. lgb_oof_probs shape:", lgb_oof_probs.shape)


Streaming output truncated to the last 5000 lines.
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with posit

# Blend deep + LightGBM OOF and compute ONE final metric set

In [10]:

# FULL MODEL OOF METRICS (deep + LGBM blended)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

alpha = 0.7  # final chosen blend weight

oof_probs = np.zeros((len(X), n_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y, groups)):
    model = fold_models[fold]
    preds = model.predict(X[va_idx], batch_size=128, verbose=0)
    oof_probs[va_idx] = preds

gid = groups  # window → sample_index
unique_ids = np.unique(gid)

sample_true = []
deep_sample_prob = []

for sid in unique_ids:
    idxs = np.where(gid == sid)[0]
    true_label = y[idxs[0]]
    mean_prob  = oof_probs[idxs].mean(axis=0)

    sample_true.append(true_label)
    deep_sample_prob.append(mean_prob)

sample_true       = np.array(sample_true)
deep_sample_prob  = np.array(deep_sample_prob)

print("Deep sample-level OOF probs shape:", deep_sample_prob.shape)
print("LGBM OOF probs shape           :", lgb_oof_probs.shape)

# Blend deep + LightGBM at sample level
blend_probs = alpha * deep_sample_prob + (1.0 - alpha) * lgb_oof_probs
blend_pred  = blend_probs.argmax(axis=1)

# Compute metrics
acc  = accuracy_score(sample_true, blend_pred)
prec = precision_score(sample_true, blend_pred, average="macro")
rec  = recall_score(sample_true, blend_pred, average="macro")
f1   = f1_score(sample_true, blend_pred, average="macro")

try:
    auc = roc_auc_score(
        tf.keras.utils.to_categorical(sample_true, n_classes),
        blend_probs,
        average="macro",
        multi_class="ovr"
    )
except:
    auc = float("nan")

print("\n==================== FULL MODEL OOF METRICS (alpha = 0.7) ====================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("=============================================================================")


Deep sample-level OOF probs shape: (661, 3)
LGBM OOF probs shape           : (661, 3)

==================== FULL MODEL OOF METRICS (alpha = 0.7) ====================
Accuracy : 0.9183
Precision: 0.8243
Recall   : 0.8163
F1-score : 0.8194
ROC-AUC  : 0.9588


In [11]:

# FULL MODEL OOF METRICS (deep + LGBM blended)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

alpha = 0.8  # final chosen blend weight

oof_probs = np.zeros((len(X), n_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y, groups)):
    model = fold_models[fold]
    preds = model.predict(X[va_idx], batch_size=128, verbose=0)
    oof_probs[va_idx] = preds

gid = groups  # window → sample_index
unique_ids = np.unique(gid)

sample_true = []
deep_sample_prob = []

for sid in unique_ids:
    idxs = np.where(gid == sid)[0]
    true_label = y[idxs[0]]
    mean_prob  = oof_probs[idxs].mean(axis=0)

    sample_true.append(true_label)
    deep_sample_prob.append(mean_prob)

sample_true       = np.array(sample_true)
deep_sample_prob  = np.array(deep_sample_prob)

print("Deep sample-level OOF probs shape:", deep_sample_prob.shape)
print("LGBM OOF probs shape           :", lgb_oof_probs.shape)

# Blend deep + LightGBM at sample level
blend_probs = alpha * deep_sample_prob + (1.0 - alpha) * lgb_oof_probs
blend_pred  = blend_probs.argmax(axis=1)

# Compute metrics
acc  = accuracy_score(sample_true, blend_pred)
prec = precision_score(sample_true, blend_pred, average="macro")
rec  = recall_score(sample_true, blend_pred, average="macro")
f1   = f1_score(sample_true, blend_pred, average="macro")

try:
    auc = roc_auc_score(
        tf.keras.utils.to_categorical(sample_true, n_classes),
        blend_probs,
        average="macro",
        multi_class="ovr"
    )
except:
    auc = float("nan")

print("\n==================== FULL MODEL OOF METRICS (alpha = 0.7) ====================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("=============================================================================")


Deep sample-level OOF probs shape: (661, 3)
LGBM OOF probs shape           : (661, 3)

==================== FULL MODEL OOF METRICS (alpha = 0.7) ====================
Accuracy : 0.9017
Precision: 0.7851
Recall   : 0.8062
F1-score : 0.7941
ROC-AUC  : 0.9549


In [12]:

# FULL MODEL OOF METRICS (deep + LGBM blended)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

alpha = 0.6  # final chosen blend weight

oof_probs = np.zeros((len(X), n_classes), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y, groups)):
    model = fold_models[fold]
    preds = model.predict(X[va_idx], batch_size=128, verbose=0)
    oof_probs[va_idx] = preds

gid = groups  # window → sample_index
unique_ids = np.unique(gid)

sample_true = []
deep_sample_prob = []

for sid in unique_ids:
    idxs = np.where(gid == sid)[0]
    true_label = y[idxs[0]]
    mean_prob  = oof_probs[idxs].mean(axis=0)

    sample_true.append(true_label)
    deep_sample_prob.append(mean_prob)

sample_true       = np.array(sample_true)
deep_sample_prob  = np.array(deep_sample_prob)

print("Deep sample-level OOF probs shape:", deep_sample_prob.shape)
print("LGBM OOF probs shape           :", lgb_oof_probs.shape)

# Blend deep + LightGBM at sample level
blend_probs = alpha * deep_sample_prob + (1.0 - alpha) * lgb_oof_probs
blend_pred  = blend_probs.argmax(axis=1)

# Compute metrics
acc  = accuracy_score(sample_true, blend_pred)
prec = precision_score(sample_true, blend_pred, average="macro")
rec  = recall_score(sample_true, blend_pred, average="macro")
f1   = f1_score(sample_true, blend_pred, average="macro")

try:
    auc = roc_auc_score(
        tf.keras.utils.to_categorical(sample_true, n_classes),
        blend_probs,
        average="macro",
        multi_class="ovr"
    )
except:
    auc = float("nan")

print("\n==================== FULL MODEL OOF METRICS (alpha = 0.7) ====================")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")
print("=============================================================================")


Deep sample-level OOF probs shape: (661, 3)
LGBM OOF probs shape           : (661, 3)

==================== FULL MODEL OOF METRICS (alpha = 0.7) ====================
Accuracy : 0.9274
Precision: 0.8575
Recall   : 0.8289
F1-score : 0.8426
ROC-AUC  : 0.9610
